## Function to Evaluate ML Models 

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(name, model, X_train, y_train, X_test, y_test, binary=True, class_names=None):
    print(f"\n=== {name} ===")

    start_train = time.time()
    model.fit(X_train, y_train)
    end_train = time.time()

    start_pred = time.time()
    y_pred = model.predict(X_test)
    end_pred = time.time()

    acc = accuracy_score(y_test, y_pred)
    train_time = end_train - start_train
    infer_time = end_pred - start_pred

    results = {
        'Model': name,
        'Accuracy': acc,
        'Training Time (s)': train_time,
        'Inference Time (s)': infer_time
    }

    if binary:
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        far  = fp / (fp + tn) if (fp + tn) > 0 else 0
        results.update({'Precision': prec, 'Recall': rec, 'F1 Score': f1, 'FAR': far})
        labels = ['Benign', 'Attack']
    else:
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        results.update({
            'Precision (weighted)': prec,
            'Recall (weighted)': rec,
            'F1 Score (weighted)': f1
        })

        try:
            y_score = model.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_score, multi_class='ovr', average='weighted')
            results['ROC-AUC (weighted)'] = auc
        except:
            results['ROC-AUC (weighted)'] = None

        # Per-class metrics
        print("\n Per-Class Metrics:")
        cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(class_names)))
        total = np.sum(cm)
        for i, label in enumerate(class_names):
            TP = cm[i, i]
            FP = cm[:, i].sum() - TP
            FN = cm[i, :].sum() - TP
            TN = total - (TP + FP + FN)
            far = FP / (FP + TN) if (FP + TN) > 0 else 0
            recall = recall_score(y_test, y_pred, labels=[i], average='macro', zero_division=0)
            f1_cls = f1_score(y_test, y_pred, labels=[i], average='macro', zero_division=0)
            print(f"  {label:<25} ➜ Recall: {recall:.3f} | F1: {f1_cls:.3f} | FAR: {far:.3f}")

        labels = list(class_names)

        # Print confusion matrix as text table
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
        print("\n Confusion Matrix Table:\n")
        print(cm_df.to_string())

    # Plot confusion matrix heatmap
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        annot_kws={"size": 14}
    )
    plt.title(f"Confusion Matrix — {name}", fontsize=16)
    plt.ylabel("True label", fontsize=14)
    plt.xlabel("Predicted label", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout(pad=3)
    plt.show()

    return results


## Function to Evaluate DL Models 

In [ ]:


def evaluate_dl_model(name, model, X_test, y_test, class_names=None):
    print(f"\n=== {name} ===")

    start_pred = time.time()
    y_pred_raw = model.predict(X_test)
    end_pred = time.time()
    infer_time = end_pred - start_pred

    # Convert predictions
    if y_pred_raw.ndim > 1 and y_pred_raw.shape[1] > 1:
        y_pred = np.argmax(y_pred_raw, axis=1)
        y_true = np.argmax(y_test, axis=1)
        is_binary = False
    else:
        y_pred = (y_pred_raw > 0.5).astype(int).flatten()
        y_true = y_test
        is_binary = True

    acc = accuracy_score(y_true, y_pred)
    results = {
        'Model': name,
        'Accuracy': acc,
        'Training Time (s)': None,
        'Inference Time (s)': infer_time
    }

    if is_binary:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        f1   = f1_score(y_true, y_pred, zero_division=0)
        far  = fp / (fp + tn) if (fp + tn) > 0 else 0
        results.update({'Precision': prec, 'Recall': rec, 'F1 Score': f1, 'FAR': far})
        labels = ['Benign', 'Attack']
    else:
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        results.update({
            'Precision (weighted)': prec,
            'Recall (weighted)': rec,
            'F1 Score (weighted)': f1
        })

        try:
            auc = roc_auc_score(y_true, y_pred_raw, multi_class='ovr', average='weighted')
            results['ROC-AUC (weighted)'] = auc
        except:
            results['ROC-AUC (weighted)'] = None

        #  Per-class metrics
        print("\n Per-Class Metrics:")
        cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
        total = np.sum(cm)
        for i, label in enumerate(class_names):
            TP = cm[i, i]
            FP = cm[:, i].sum() - TP
            FN = cm[i, :].sum() - TP
            TN = total - (TP + FP + FN)
            far = FP / (FP + TN) if (FP + TN) > 0 else 0
            recall_i = recall_score(y_true, y_pred, labels=[i], average='macro', zero_division=0)
            f1_i = f1_score(y_true, y_pred, labels=[i], average='macro', zero_division=0)
            print(f"  {label:<25} ➜ Recall: {recall_i:.3f} | F1: {f1_i:.3f} | FAR: {far:.3f}")

        labels = list(class_names)

    # Summary
    print(f"\nAccuracy:               {acc:.4f}")
    print(f"⏱ Inference Time:       {infer_time:.2f} sec")
    if is_binary:
        print(f"Precision:              {prec:.4f}")
        print(f"Recall:                 {rec:.4f}")
        print(f"F1 Score:               {f1:.4f}")
        print(f"False Alarm Rate (FAR): {far:.4f}")
    else:
        print(f"Precision (weighted):   {prec:.4f}")
        print(f"Recall (weighted):      {rec:.4f}")
        print(f"F1 Score (weighted):    {f1:.4f}")
        if results['ROC-AUC (weighted)'] is not None:
            print(f" ROC-AUC (weighted):  {results['ROC-AUC (weighted)']:.4f}")
        else:
            print("ROC-AUC could not be computed.")

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(cm,
                         index=[f"True {label}" for label in labels],
                         columns=[f"Pred {label}" for label in labels])
    print("\n Confusion Matrix Table (Text):")
    print(cm_df.to_string())

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                annot_kws={"size": 14})
    plt.title(f"Confusion Matrix — {name}", fontsize=16)
    plt.ylabel("True label", fontsize=14)
    plt.xlabel("Predicted label", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout(pad=3)
    plt.show()

    return results


## Data Preprocessing

In [ ]:
# Import Data

import pandas as pd

file_path = r'C:\Users\user\CSE-CIC-IDS2018-Full\clean_sample.csv'
df = pd.read_csv(file_path)


### Data Preprocessing for Multiclass Classification 

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

#FOR MULTICLASS 

# Encode the 'Label' column
label_encoder = LabelEncoder()
df['Label_encoded'] = label_encoder.fit_transform(df['Label'])

# Separate features and target
X = df.drop(columns=['Label', 'Label_encoded'])  # remove original Label
y = df['Label_encoded']

# Stratified train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Scale features 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)


### Data Preprocessing for Binary Classification 

In [ ]:

# Encode the 'Label' column 
df['Label_binary'] = df['Label'].apply(lambda x: 0 if x == 'Benign' else 1)

# Split features and target
X_bin = df.drop(columns=['Label', 'Label_encoded', 'Label_binary'])
y_bin = df['Label_binary']

# Stratified train/val/test split
Xb_train, Xb_temp, yb_train, yb_temp = train_test_split(
    X_bin, y_bin, test_size=0.30, stratify=y_bin, random_state=42)

Xb_val, Xb_test, yb_val, yb_test = train_test_split(
    Xb_temp, yb_temp, test_size=0.5, stratify=yb_temp, random_state=42)

# Scale 
scaler_bin = StandardScaler()
Xb_train_scaled = scaler_bin.fit_transform(Xb_train)
Xb_val_scaled   = scaler_bin.transform(Xb_val)
Xb_test_scaled  = scaler_bin.transform(Xb_test)


## Class Balancing 

### For Multiclass Classification

In [ ]:
from collections import Counter
import numpy as np
import pandas as pd

# Thresholds
LARGE_THRESHOLD = 30_000
MEDIUM_THRESHOLD = 10_000
RARE_THRESHOLD = 1_000
RARE_TARGET_SIZE = 300

# Reconstruct labeled DataFrame
y_train_labels = label_encoder.inverse_transform(y_train)
X_df = pd.DataFrame(X_train_scaled)
X_df['Label'] = y_train_labels

X_balanced = []
y_balanced = []

for label, group in X_df.groupby('Label'):
    count = len(group)
    X_vals = group.drop(columns='Label').values
    y_vals = group['Label'].values

    # Benign: keep it as it is 
    if label == 'Benign':
        X_res = X_vals
        y_res = y_vals

    # Large: keep it as it is 
    elif count > LARGE_THRESHOLD:
        X_res = X_vals
        y_res = y_vals

    # Medium: upsample to 30k (repititions)
    elif count > MEDIUM_THRESHOLD:
        target = 30_000
        reps = target // count + 1
        X_upsampled = np.tile(X_vals, (reps, 1))[:target]
        y_upsampled = np.tile(y_vals, reps)[:target]
        X_res = X_upsampled
        y_res = y_upsampled

    # Small: upsample to x2
    elif count > RARE_THRESHOLD:
        target = count * 2
        reps = target // count + 1
        X_upsampled = np.tile(X_vals, (reps, 1))[:target]
        y_upsampled = np.tile(y_vals, reps)[:target]
        X_res = X_upsampled
        y_res = y_upsampled

    # Rare: upsample to 300
    else:
        target = RARE_TARGET_SIZE
        reps = target // count + 1
        X_upsampled = np.tile(X_vals, (reps, 1))[:target]
        y_upsampled = np.tile(y_vals, reps)[:target]
        X_res = X_upsampled
        y_res = y_upsampled

    X_balanced.append(X_res)
    y_balanced.append(y_res)

X_final = np.vstack(X_balanced)
y_final = np.concatenate(y_balanced)
y_final_encoded = label_encoder.transform(y_final)

# Print final class distribution
print("\n Final class distribution:")
for k, v in Counter(y_final).items():
    print(f"{k:<30}: {v}")


### For Binary Classification

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

def print_distribution(title, y_data, encoder=None):
    print(f"\n {title} Class Distribution:")
    counts = Counter(y_data)
    for label, count in sorted(counts.items()):
        if encoder:
            name = encoder.inverse_transform([label])[0]
        else:
            name = "Benign" if label == 0 else "Attack"
        print(f"  {name:<25} ➜ {count:,} samples")

# Random undersampling for majority class "Benign"
print_distribution("Binary - Before Balancing", yb_train)

rus = RandomUnderSampler(random_state=42)
Xb_train_bal, yb_train_bal = rus.fit_resample(Xb_train_scaled, yb_train)

print_distribution("Binary - After Balancing", yb_train_bal)
print("Balanced Binary Training Shape:", Xb_train_bal.shape)




## Train ML Models on the Balanced Data 

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Define ML Models 
models = {
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}

# Evaluate 
all_results = []

for name, model in models.items():
    print(f"\n Multiclass — {name}")
    result_multi = evaluate_model(
        name=f"{name} (Multiclass)",
        model=model,
        X_train=X_final,                 
        y_train=y_final_encoded,         
        X_test=X_test_scaled,            
        y_test=y_test,                   
        binary=False,
        class_names=label_encoder.classes_
    )
    all_results.append(result_multi)

    print(f"\n Binary — {name}")
    result_bin = evaluate_model(
        name=f"{name} (Binary)",
        model=model,
        X_train=Xb_train_bal,            
        y_train=yb_train_bal,
        X_test=Xb_test_scaled,
        y_test=yb_test,
        binary=True
    )
    all_results.append(result_bin)

results_df = pd.DataFrame(all_results)
display(results_df)


## Train 1D CNN Model on the Balanced Data

### CNN - Multiclass Classification 

In [ ]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import random
import numpy as np
import tensorflow as tf

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


# One-hot encode target for multiclass classification
y_train_cat = to_categorical(y_final_encoded)         # Balanced 
y_val_cat = to_categorical(y_val)                     # original 
y_test_cat = to_categorical(y_test)                   # original 

# Reshape input for Conv1D: (samples, time_steps, features)
X_train_cnn = X_final.reshape(X_final.shape[0], X_final.shape[1], 1) 
X_val_cnn   = X_val_scaled.reshape(X_val_scaled.shape[0], X_val_scaled.shape[1], 1)
X_test_cnn  = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)


# Define CNN model
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(y_train_cat.shape[1], activation='softmax')  # number of classes
])

# Compile
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Early stopping to prevent overfitting
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train model
history = model.fit(
    X_train_cnn, y_train_cat,
    validation_data=(X_val_cnn, y_val_cat),
    epochs=30,
    batch_size=128,
    callbacks=[es],
    verbose=1
)

import matplotlib.pyplot as plt

# Accuracy Plot
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

evaluate_dl_model(
    name="CNN Multiclass",
    model=model,
    X_test=X_test_cnn,
    y_test=y_test_cat,
    class_names=label_encoder.classes_  
)

### CNN - Binary Classification 

In [ ]:

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Reshape input
Xb_train_cnn = Xb_train_bal.reshape(Xb_train_bal.shape[0], Xb_train_bal.shape[1], 1)   
Xb_val_cnn   = Xb_val_scaled.reshape(Xb_val_scaled.shape[0], Xb_val_scaled.shape[1], 1)
Xb_test_cnn  = Xb_test_scaled.reshape(Xb_test_scaled.shape[0], Xb_test_scaled.shape[1], 1)

model_bin_cnn = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(Xb_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid') 
])

model_bin_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

es_bin = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_bin_cnn = model_bin_cnn.fit(
    Xb_train_cnn, yb_train_bal,          
    validation_data=(Xb_val_cnn, yb_val),
    epochs=30,
    batch_size=128,
    callbacks=[es_bin],
    verbose=1
)

results_bin_cnn = evaluate_dl_model(
    name="CNN Binary (Balanced Train)",
    model=model_bin_cnn,
    X_test=Xb_test_cnn,      
    y_test=yb_test,
    class_names=['Benign', 'Attack']
)

pd.DataFrame([results_bin_cnn])

plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_bin_cnn.history['accuracy'], label='Train Acc')
plt.plot(history_bin_cnn.history['val_accuracy'], label='Val Acc')
plt.title('CNN Binary Model Accuracy (Balanced)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_bin_cnn.history['loss'], label='Train Loss')
plt.plot(history_bin_cnn.history['val_loss'], label='Val Loss')
plt.title('CNN Binary Model Loss (Balanced)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


## Train LSTM Model on Balanced Data 

### LSTM - Multiclass Classification 

In [ ]:
import numpy as np
import tensorflow as tf
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

y_train_cat = to_categorical(y_final_encoded)     
y_val_cat   = to_categorical(y_val)              
y_test_cat  = to_categorical(y_test)             

X_train_lstm = X_final.reshape(X_final.shape[0], 1, X_final.shape[1])           
X_val_lstm   = X_val_scaled.reshape(X_val_scaled.shape[0], 1, X_val_scaled.shape[1])
X_test_lstm  = X_test_scaled.reshape(X_test_scaled.shape[0], 1, X_test_scaled.shape[1])

model_lstm = Sequential([
    LSTM(128, input_shape=(1, X_train_lstm.shape[2]), return_sequences=False),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(y_train_cat.shape[1], activation='softmax') 
])

model_lstm.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

es_lstm = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_lstm = model_lstm.fit(
    X_train_lstm, y_train_cat,
    validation_data=(X_val_lstm, y_val_cat),
    epochs=30,
    batch_size=128,
    callbacks=[es_lstm],
    verbose=1
)

results_lstm_multi = evaluate_dl_model(
    name="LSTM Multiclass",
    model=model_lstm,
    X_test=X_test_lstm,
    y_test=y_test_cat,
    class_names=label_encoder.classes_  
)

plt.figure(figsize=(12, 4))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_lstm.history['accuracy'], label='Train Accuracy')
plt.plot(history_lstm.history['val_accuracy'], label='Val Accuracy')
plt.title('LSTM Multiclass Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_lstm.history['loss'], label='Train Loss')
plt.plot(history_lstm.history['val_loss'], label='Val Loss')
plt.title('LSTM Multiclass Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


### LSTM - Binary Classification 

In [ ]:

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

Xb_train_lstm = Xb_train_bal.reshape(Xb_train_bal.shape[0], 1, Xb_train_bal.shape[1]) 
Xb_val_lstm   = Xb_val_scaled.reshape(Xb_val_scaled.shape[0], 1, Xb_val_scaled.shape[1])
Xb_test_lstm  = Xb_test_scaled.reshape(Xb_test_scaled.shape[0], 1, Xb_test_scaled.shape[1])

model_bin_lstm = Sequential([
    LSTM(64, input_shape=(1, Xb_train_bal.shape[1]), return_sequences=False),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_bin_lstm.compile(optimizer='adam',
                       loss='binary_crossentropy',
                       metrics=['accuracy'])

es_bin_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_bin_lstm = model_bin_lstm.fit(
    Xb_train_lstm, yb_train_bal,
    validation_data=(Xb_val_lstm, yb_val),  
    batch_size=64,
    callbacks=[es_bin_lstm],
    verbose=1
)

results_bin_lstm = evaluate_dl_model(
    name="LSTM Binary",
    model=model_bin_lstm,
    X_test=Xb_test_lstm,    
    y_test=yb_test,
    class_names=['Benign', 'Attack']
)

plt.figure(figsize=(12, 4))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_bin_lstm.history['accuracy'], label='Train Accuracy')
plt.plot(history_bin_lstm.history['val_accuracy'], label='Val Accuracy')
plt.title('Binary LSTM Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_bin_lstm.history['loss'], label='Train Loss')
plt.plot(history_bin_lstm.history['val_loss'], label='Val Loss')
plt.title('Binary LSTM Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

pd.DataFrame([results_bin_lstm])
